# 🌲 Needle 3 PyTorch 模型架構實驗室

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Child-pi/needle/blob/pytorch_experiment/pytorch_experiment.ipynb)

歡迎來到 **Needle 3 PyTorch 實驗室**！

Needle 3 原生官方使用 Google JAX/Flax 進行預訓練與 LoRA 微調，並編譯為專有 `.cact` 二進制檔由 C 引擎執行。
本實驗室展示了 **Needle 3 核心架構的 PyTorch 完整移植實現**：
- ⚡ **Monarch Hadamard MLP**：以塊對角與 Hadamard 旋轉矩陣取代傳統笨重 FFN
- 🎯 **GQA (Grouped-Query Attention)**：因果卷積抽頭 + RoPE 旋轉位置編碼
- 🛡️ **ZCRMSNorm**：Zero-Centered RMSNorm 歸一化層
- 📊 **Calibrated Confidence Head**：模型內建置信度預測頭
- 🪜 **Laddered Depth Slicing**：可隨意指定 2、4、8、16、20 層任意階梯深度

--- 
## 步驟 1：下載 pytorch_experiment 分支程式庫

直接從 GitHub 克隆包含 PyTorch 實現的 `pytorch_experiment` 分支：

In [ ]:
# 下載並切換至 pytorch_experiment 分支
!git clone -b pytorch_experiment https://github.com/Child-pi/needle.git
%cd needle

import torch
print(f"✅ PyTorch 載入成功！版本: {torch.__version__}, 是否有 CUDA GPU: {torch.cuda.is_available()}")

--- 
## 步驟 2：初始化 PyTorch 版 Needle 3 模型

我們可以自由切換不同的階梯深度（例如 4 層、8 層或 20 層）：

In [ ]:
from needle.pytorch import NeedleConfig, NeedleForCausalLM

# 配置 4 層 (約 29M 參數) 的輕量化邊緣子網路
config = NeedleConfig(
    vocab_size=16384,
    d_model=768,
    num_heads=12,
    num_kv_heads=2,
    num_layers=4,        # 可自由設定為 2, 4, 8, 16, 20
    qk_head_dim=48,
    v_head_dim=64,
    qkv_conv_taps=3,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = NeedleForCausalLM(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"📊 模型建立成功！總參數量: {total_params / 1e6:.2f} M")
print(f"💻 運行裝置: {device}")

--- 
## 步驟 3：模型前向傳播 (Forward Pass) 與置信度預測

測試輸入 Token 序列，計算 Logits 與模型內建的 Calibrated Confidence 分數：

In [ ]:
# 模擬 Batch=2, 長度=16 的輸入 Token ID 序列
input_ids = torch.randint(0, config.vocab_size, (2, 16), device=device)

with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs["logits"]
    confidence = outputs["confidence"]

print("=== 前向傳播計算結果 ===")
print(f"- 輸出 Logits 形狀: {list(logits.shape)} (批次, 序列長度, 詞表大小)")
print(f"- 內建校準置信度 (Confidence): {confidence.cpu().tolist()}")

--- 
## 步驟 4：自回歸文字生成 (Autoregressive Generation)

使用 PyTorch 實現的 `generate` 進行自回歸 Greedy 解碼生成：

In [ ]:
# 給定起始提示詞 Token
prompt_tokens = torch.tensor([[101, 2045, 12, 88]], dtype=torch.long, device=device)
print(f"起始 Prompt Token IDs: {prompt_tokens[0].tolist()}")

# 自回歸生成接下來的 12 個 Token
generated = model.generate(prompt_tokens, max_new_tokens=12, temperature=0.0)

print(f"生成的完整 Token 序列: {generated[0].tolist()}")
print(f"總生成長度: {generated.shape[1]} tokens")

--- 
## 步驟 5：反向傳播與微調訓練 (PyTorch Backprop & Training Step)

驗證在 PyTorch 下所有自定義層（包含 Monarch Hadamard MLP 與因果卷積）的梯度回傳與反向傳播：

In [ ]:
import torch.nn.functional as F

# 初始化 AdamW 優化器
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)

# 模擬訓練步驟
targets = torch.randint(0, config.vocab_size, (2, 16), device=device)
outputs = model(input_ids)
logits = outputs["logits"]

# 計算交叉熵損失
loss = F.cross_entropy(logits.view(-1, config.vocab_size), targets.view(-1))

optimizer.zero_grad()
loss.backward()

# 梯度裁剪與權重更新
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()

print(f"✓ 交叉熵損失 Cross-Entropy Loss: {loss.item():.4f}")
print(f"✓ 總梯度範數 Gradient Norm: {grad_norm.item():.4f}")
print("🎉 PyTorch 訓練更新步執行成功！")

--- 
## 步驟 6：探索 Monarch Hadamard MLP 結構

觀察 Needle 特有的 Monarch Hadamard 前饋層參數：

In [ ]:
block0 = model.model.layers[0]
hada = block0.hadamard_mlp

print("Monarch Hadamard MLP 參數檢視：")
print("- 旋轉因子矩陣 w1a 形狀:", hada.w1a.shape)
print("- 旋轉因子矩陣 w1b 形狀:", hada.w1b.shape)
print("- 縮放對角向量 d1 形狀:", hada.d1.shape)
print("- 條件向量 cond_v 形狀:", hada.cond_v.shape)
print("- 置換映射 p1 長度:", len(hada.p1))

--- 
## 總結

透過此實驗室，我們驗證了：
1. **架構可行性**：Needle 3 的 Laddered Simple Attention Network 完全可以用標準 PyTorch 實現。
2. **全流程相容**：前向推論、自回歸生成、反向傳播訓練與 AdamW 優化器皆無縫運作。
3. **研究與自定義**：現在您可以基於此 PyTorch 程式庫進行自定義架構實驗、量化測試或與 PyTorch 生態系（如 Hugging Face、vLLM、TensorRT-LLM）整合！